# SPACESHIP TITANIC CHALLENGE  

---


---

## Table of Contents:  
1. Introduction
2. Setup
3. Missing Values
4. Exploring the Data
5. Prediction
6. Conclusion & Next Steps


## 1. Introduction
---
This notebook explores the Spaceship Titanic dataset from Kaggle through manual 
exploratory data analysis (EDA) using only pandas, and a hand-built scorecard model, 
achieving 78.7% accuracy on Kaggle without the use of Machine Learning libraries.

The Spaceship Titanic challenge aims to determine the fate of each pasenger who travels on board the ship.
Given passenger data (age, cabin, spending habits, home planet...), we try to predict which passengers are likely to disapper in a space anomaly.

---

Dataset description provided:
| Column | Description |
|---|---|
| PassengerId | Unique ID in format GroupNumber_PersonNumber |
| HomePlanet | Planet the passenger departed from |
| CryoSleep | Whether the passenger was traveling in a cryo pod |
| Cabin | Cabin number in format Deck/Cabin/Side |
| Destination | Planet the passenger is travelling to |
| Age | Age of the passenger |
| VIP | Whether the passenger paid for VIP service |
| RoomService | Amount spent at the room service facility |
| FoodCourt | Amount spent at the food court |
| ShoppingMall | Amount spent at the shopping mall |
| Spa | Amount spent at the spa |
| VRDeck | Amount spent at the VR deck |
| Name | Passenger name |
| Transported | Whether the passenger was transported by the anomaly|

## 2. Setup
---

In [1]:
import pandas as pd
data = pd.read_csv('train.csv', index_col='PassengerId')

## 3. Missing Values
---
Let's get some info about the dataset first:

In [2]:
data.info()
data.head()

<class 'pandas.DataFrame'>
Index: 8693 entries, 0001_01 to 9280_02
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   HomePlanet    8492 non-null   str    
 1   CryoSleep     8476 non-null   object 
 2   Cabin         8494 non-null   str    
 3   Destination   8511 non-null   str    
 4   Age           8514 non-null   float64
 5   VIP           8490 non-null   object 
 6   RoomService   8512 non-null   float64
 7   FoodCourt     8510 non-null   float64
 8   ShoppingMall  8485 non-null   float64
 9   Spa           8510 non-null   float64
 10  VRDeck        8505 non-null   float64
 11  Name          8493 non-null   str    
 12  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(2), str(4)
memory usage: 891.4+ KB


,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
PassengerId,,,,,,,,,,,,,
0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


We see that all columns have a few missing values, first thing we do will fill them up. We'll use the median for numerical data, and the mode for strings. Let's make a quick function to do it easily.

In [3]:
def fill_missing(data):
    """ Filling the missing values for each column, using mode and median for the time being. 
    Important note: filling with mode will create groups of ~200 people with the same data.
    Will find a better way to handle missing values after learning more."""
    for column in data.columns:
        if data[column].dtype in ('str', 'object', 'bool'):
            data[column] = data[column].fillna(data[column].mode()[0])
        else:
            data[column] = data[column].fillna(data[column].median())
    return data


fill_missing(data)
data.isnull().sum()

HomePlanet      0
CryoSleep       0
Cabin           0
Destination     0
Age             0
VIP             0
RoomService     0
FoodCourt       0
ShoppingMall    0
Spa             0
VRDeck          0
Name            0
Transported     0
dtype: int64

## 4. Exploring the Data
---
Now let's see what we can get from the data.

In [4]:
print(f'Passenger count: {len(data.index)}')
print((data['HomePlanet'].value_counts(normalize=True) * 100).round())
print((data.groupby('HomePlanet').Transported.mean() * 100).round())


Passenger count: 8693
HomePlanet
Earth     55.0
Europa    25.0
Mars      20.0
Name: proportion, dtype: float64
HomePlanet
Earth     43.0
Europa    66.0
Mars      52.0
Name: Transported, dtype: float64


Earth passengers make up 55% of the ship, but have the lowest Transporation rate(43%).  
On the other hand, 66% of Europa passengers were affected by the singularity.  

In [5]:
print(data['VIP'].value_counts(normalize=True) * 100)
print(data.groupby('VIP').Transported.mean() * 100)

VIP
False    97.710802
True      2.289198
Name: proportion, dtype: float64
VIP
False    50.647516
True     38.190955
Name: Transported, dtype: float64


Only 2% of the passengers had VIP status, for a Transporation rate of 38%. It's better than non-VIP passengers, but since it's such a small number of people,the impact of this column isn't significant.

In [6]:
data.groupby(['HomePlanet', 'CryoSleep'])['Transported'].agg(['mean', 'count'])

mean  count
HomePlanet CryoSleep                 
Earth      False      0.323371   3346
           True       0.667124   1457
Europa     False      0.412295   1220
           True       0.989023    911
Mars       False      0.284404   1090
           True       0.911809    669

99% Transporation rate for Europa passengers in cryosleep, and 91% for Mars passengers in Cryosleep. The pattern is very strong here. Earth is at 67% so will need breaking it down.
The pattern appears clear so far: Europa and Mars have higher risks, and CryoSleep greatly increases the risks across all planets.

In [7]:
data.groupby(['HomePlanet', 'Destination'])['Transported'].agg(['mean', 'count'])

mean  count
HomePlanet Destination                   
Earth      55 Cancri e    0.511789    721
           PSO J318.5-22  0.501374    728
           TRAPPIST-1e    0.393560   3354
Europa     55 Cancri e    0.689616    886
           PSO J318.5-22  0.736842     19
           TRAPPIST-1e    0.635400   1226
Mars       55 Cancri e    0.611399    193
           PSO J318.5-22  0.448980     49
           TRAPPIST-1e    0.514173   1517

Breaking donwn Destination and Transported rate by home planet.   
Earth > TRAPPIST and Mars > PSO seem to diverge from the average. 
- Mars > PSO is weak, only 49 people, not worth looking into.
- Earth > TRAPPIST is 3300 people, the difference could perhaps mean something, though 60/40 isn't strong.

In [8]:
data['AgeGroup'] = pd.cut(data['Age'], bins=[0,5,10,16,25,40,60,100]) 
pd.crosstab([data['HomePlanet'], data['CryoSleep']], data['AgeGroup']).style.format("{:,}")

In [9]:
print('Transporation rate for each Age Group split by Cryo Sleep')
pd.crosstab(data['CryoSleep'], data['AgeGroup'], values=data['Transported'], aggfunc='mean').style.format("{:.1%}")

Transporation rate for each Age Group split by Cryo Sleep


AgeGroup,"(0, 5]","(5, 10]","(10, 16]","(16, 25]","(25, 40]","(40, 60]","(60, 100]"
CryoSleep,,,,,,,
False,71.4%,59.3%,31.1%,31.3%,29.7%,33.2%,27.5%
True,73.9%,59.5%,83.7%,79.3%,84.9%,83.6%,88.7%


This confirms that cryosleep is a very strong indicator for Transportation.
- People from Earth were more likely to have babies in Cryo Sleep. Overall, Earthians were mostly in the 16-25 year old group.
- People from Europa were less likely to have babies and young children in Cryo Sleep, but more likely to have teenagers there. Overall they were mostly in the 25-40 year old group.  
More useful information:
- Young children below 10 are barely affected by Cryo Sleep, the Transporation rate is roughly the same.

In [10]:
data.groupby('HomePlanet')[['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']].mean().style.format("{:,.2f}")


,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
HomePlanet,,,,,
Earth,136.51,139.60,130.87,143.42,141.02
Europa,142.78,"1,470.76",147.37,830.15,860.56
Mars,541.58,53.19,302.13,107.97,46.39


- Earthians don't stand out in any luxury activity.
- Europans are very big spenders on the Food Court, Spa, and VR Deck.
- Marsians are big spenders on RoomService & Shopping Mall.

In [11]:
data.groupby('Transported')[['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']].mean().style.format("{:,.2f}")


,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
Transported,,,,,
False,380.43,375.17,163.92,552.35,532.29
True,61.90,520.65,175.15,60.39,67.60


- Heavy Spenders on Room Service, Spa, and VR Deck tend to be a lot safer.
- Food Court & Shopping Mall were in bigger danger, especially the Food Court.  

This can be useful to determine which Europans & Marsians are likely to be transported, depending on which servives they use. Meaningless for Earthians.  

In [12]:
data['LastName'] = data['Name'].map(lambda n: n.rsplit()[-1])
data['FamilySize'] = data['LastName'].map(data['LastName'].value_counts())
data.groupby('FamilySize')['Transported'].mean()

FamilySize
1      0.583770
2      0.493750
3      0.534364
4      0.503906
5      0.509375
6      0.499037
7      0.481840
8      0.454044
9      0.482540
10     0.510000
11     0.505455
12     0.428571
13     0.461538
14     0.482143
15     0.600000
16     0.375000
18     0.388889
207    0.502415
Name: Transported, dtype: float64

Trying to detect families, grouping people by last name. No pattern detected, everything is around 50%
Let's try a different approach. Passenger ID should tell us if the passengers traveled in a group. We can also group people by cabin number.

In [13]:
data['PassengerGroup'] = data.index.map(lambda g: g.split('_')[0]) # Creates a column for each passenger with their group number
data['GroupSize'] = data['PassengerGroup'].map(data['PassengerGroup'].value_counts()) # Creates a column with the group size for each passenger
data[data['GroupSize'] > 1].groupby('PassengerGroup')['Transported'].mean().value_counts() 
# Calculates how many times groups of people were transported together, for groups of more than 1 person.


Transported
0.500000    410
1.000000    378
0.000000    237
0.666667    139
0.333333     98
0.750000     35
0.250000     16
0.600000     16
0.800000     13
0.400000     10
0.571429      9
0.833333      9
0.428571      8
0.200000      7
0.714286      5
0.375000      5
0.166667      4
0.285714      4
0.857143      4
0.625000      2
0.125000      2
0.142857      1
Name: count, dtype: int64

The interesting Data is if groups of people were transported together, so the 1.0 and the 0.00 values on this table.
In total 615 groups were transported or remained safe together, out of a total of 1400 groups. No pattern detected here.

In [14]:
print(data.groupby('GroupSize')['Transported'].agg(['mean', 'count']))

               mean  count
GroupSize                 
1          0.452445   4805
2          0.538050   1682
3          0.593137   1020
4          0.640777    412
5          0.592453    265
6          0.614943    174
7          0.541126    231
8          0.394231    104


In [15]:
pd.crosstab(data['GroupSize'], data['CryoSleep'], normalize='index').style.format("{:,.0%}")

CryoSleep,False,True
GroupSize,,
1,69%,31%
2,62%,38%
3,59%,41%
4,61%,39%
5,57%,43%
6,59%,41%
7,54%,46%
8,64%,36%


- People traveling alone have slightly lower than average chances to be Transported.
- There is a sweet spot for groups of 3 to 6 people with a Transporation rate around 60%.
- We see that groups have a higher percentage of cryo sleepers, which partly explains their higher transporation rate. Cryo Sleep is again the determining factor.  

To be thorough, we will now group people living in the same cabin. We do the same as we did for groups, but applied to cabin instead.  

In [16]:
data['CabinNumber'] = data.Cabin.map(lambda p: str(p.split('/')[1]) + '_' + str(p.split('/')[0]))
data['CabinSize'] = data['CabinNumber'].map(data['CabinNumber'].value_counts())
data[data['CabinSize'] > 1].groupby('CabinNumber')['Transported'].mean().value_counts()

Transported
0.500000    889
0.000000    558
1.000000    519
0.666667    177
0.333333    129
0.750000     63
0.250000     33
0.600000     25
0.800000     25
0.400000     19
0.200000     12
0.571429      9
0.833333      9
0.714286      7
0.857143      6
0.428571      5
0.166667      4
0.625000      3
0.875000      3
0.142857      3
0.285714      1
0.125000      1
0.700000      1
0.444444      1
0.636364      1
0.375000      1
0.888889      1
0.495192      1
Name: count, dtype: int64

No pattern detected here. Less than half of the cabins in the ship had their occupants share the same fate.  


In [17]:
data['Side'] = data.Cabin.map(lambda p: p.rsplit('/')[-1]) # Extracts the Side from Cabin column
data['Deck'] = data.Cabin.map(lambda p: p[:1]) # Extracts the Deck from Cabin column

pd.crosstab([data['Deck'], data['Side']], data['Transported'], normalize='index').style.format("{:,.0%}")

Now we check the decks and sides of the ship.
- Starboard side (right), has consistent higher transportation rates. It's clear the ship was hit harder on this side. The difference isn't huge, but worth keeping track of.
- Deck B is particularly dangerous. Deck C as well but mostly on the right side.
- Deck E, T and G's left side are the safest parts of the ship.

In [18]:
pd.crosstab([data['HomePlanet'], data['Side']], data['Deck']).style.format("{:,.0f}")

This data shows that passengers from different planets live on separated decks.
- Europans are mostly on decks A, B & C
- Martians are mostly on decks D, E & F
- Earthians are mostly on deck F, G and a few on E
- T is meaningless, only 5 people.

In [19]:
data.groupby(['Deck', 'HomePlanet'])['Transported'].mean()
pd.crosstab([data['Deck'], data['HomePlanet']], data['Transported'], normalize = 'index').style.format("{:,.0%}")

This gives us the transporation rate for passengers of each home planet, on each of the decks.
- Europans even on deck G had higher than average transporation rates.
- Marsians on Deck F had high transporation rates, while Earthians didn't.

In [20]:
pd.crosstab([data['Deck'], data['CryoSleep']], data['Transported'], normalize='index').style.format("{:,.0%}")
# Calculates the transporation rare for people in cryosleep on each deck

This is a very interesting table, because it gives us very good percentages for the transported rate depending on the deck and cryosleep.  
Except for Deck E and G, passengers in cryosleep were almost certain to get Transported. 

## 5. Prediction
---

This is enough EDA to make a prediction now. Without using any library, we will try to make a small decision tree by hand, assigning points and weights to each relevant signal.  

Main signals: CryoSleep, luxury spending, deck, home planet, side, age for young children

Difficulty is finding the right weights for each element  


In [21]:
avg_food = data['FoodCourt'].mean()
avg_shop = data['ShoppingMall'].mean()
avg_spa = data['Spa'].mean()
avg_vr = data['VRDeck'].mean()
avg_room = data['RoomService'].mean()

def prediction(data):
    survival = 0
    if data['Side'] == 'S': survival -=1
    if data['CryoSleep'] == True: survival -= 1
    if data['HomePlanet'] == 'Europa': survival -= 1
    if data['Deck'] in ('B', 'C'): survival -= 1
    if data['Age'] < 10: survival -= 1
    if data['FoodCourt'] > (avg_food * 3): survival -=1
    if data['ShoppingMall'] > (avg_shop * 2): survival -=1
    if data['Spa'] > avg_spa: survival +=3
    if data['VRDeck'] > avg_vr: survival +=3
    if data['RoomService'] > avg_room: survival +=2
    if survival < 0:
        return True
    else:
        return False
    
data['Prediction'] = data.apply(prediction, axis=1)

print((data['Prediction'] == data['Transported']).mean())

0.7823536178534453


Creating the submission csv file:

In [22]:
testdata = pd.read_csv('test.csv')  
fill_missing(testdata)
testdata['Side'] = testdata.Cabin.map(lambda p: p.rsplit('/')[-1])
testdata['Deck'] = testdata.Cabin.map(lambda p: p[:1])

testdata['Transported'] = testdata.apply(prediction, axis=1)

pd.DataFrame({
    'PassengerId': testdata['PassengerId'],
    'Transported': testdata['Transported']
}).to_csv('submission.csv', index=False)

## 6. Conclusion & Next Steps

---

Key findings:
- CryoSleep is the main predictor
- Luxury spending (Spa, VRDeck, RoomService) strongly predicts non-transport
- FoodCourt spending goes in the opposite direction
- Age < 10 is its own independant signal
- Deck, Home Planet, and Ship side increase probabilities to be transported
- Group and cabin analysis showed no strong pattern beyond CryoSleep

Manual prediction achieves 78.7% score.

Next: learn scikit-learn to improve prediction
